# Environment Setup

In [ ]:
import sys, os, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
print('Working directory:', os.getcwd())
!pip install -q --force-reinstall "torch==2.10.0" --index-url https://download.pytorch.org/whl/cu128

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Working directory: /content
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 81.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 200.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 115.7 MB/s eta 0:00:00
     ━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.9/706.8 MB 100.5 MB/s eta 0:00:06

In [ ]:
import torch, shutil
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
nvcc = shutil.which('nvcc')
print('nvcc:', nvcc if nvcc else 'not found')

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA L4
nvcc: /usr/local/cuda/bin/nvcc


# Clone Repo

In [ ]:
import os
REPO_URL  = 'https://github.com/jeromereddy9/Investigating-Contrastive-Regularisation-in-State-Space-Model-Based-Sequential-Recommender-Systems'
REPO_PATH = '/content/Investigating-Contrastive-Regularisation-in-State-Space-Model-Based-Sequential-Recommender-Systems'
if not os.path.exists(REPO_PATH):
    os.system(f'git clone {REPO_URL}')
os.chdir(REPO_PATH)
print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

Working directory: /content/Investigating-Contrastive-Regularisation-in-State-Space-Model-Based-Sequential-Recommender-Systems
Contents: ['.git', 'src', '.idea', '.gitignore', '.gitmodules', 'LICENSE']


# Install Dependencies

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'numpy==1.26.4', 'recbole==1.2.0'], check=True)
print('recbole installed')

recbole installed


In [ ]:
import torch
!pip install -U pip setuptools wheel
!pip install ninja packaging
!pip install causal-conv1d --no-build-isolation
import causal_conv1d
print("causal-conv1d:", causal_conv1d.__version__)
if torch.cuda.is_available():
    print("GPU detected — installing Mamba dependencies...")
    !pip install mamba-ssm==2.3.1 --no-build-isolation
    import mamba_ssm
    print("mamba-ssm version:", mamba_ssm.__version__)
    print("causal-conv1d installed successfully")

else:
    print("No GPU — skipping mamba-ssm")

causal-conv1d: 1.6.2.post1
GPU detected — installing Mamba dependencies...
mamba-ssm version: 2.3.1
causal-conv1d installed successfully


# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Copy Datasets from Drive

In [ ]:
import os, shutil

DRIVE_DATASET_ROOT   = '/content/drive/MyDrive/preprocessed'
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/checkpoints'
DRIVE_RESULTS_DIR    = '/content/drive/MyDrive/results'
LOCAL_DATASET_ROOT   = os.path.join(REPO_PATH, 'src/datasets/preprocessed')

os.makedirs(LOCAL_DATASET_ROOT,   exist_ok=True)
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR,    exist_ok=True)

DATASETS = [
    'amazon_videogames',
    'amazon_toys_and_games',
    'movielens_1m',
    'lastfm_1k',
]

for dataset in DATASETS:
    src  = os.path.join(DRIVE_DATASET_ROOT, dataset)
    dest = os.path.join(LOCAL_DATASET_ROOT, dataset)
    if not os.path.exists(dest):
        shutil.copytree(src, dest)
        print(f'Copied: {dataset}')
    else:
        print(f'Already exists: {dataset}')

print('\nCheckpoint dir:', DRIVE_CHECKPOINT_DIR)
print('Results dir   :', DRIVE_RESULTS_DIR)

Copied: amazon_videogames
Copied: amazon_toys_and_games
Copied: movielens_1m
Copied: lastfm_1k

Checkpoint dir: /content/drive/MyDrive/checkpoints
Results dir   : /content/drive/MyDrive/results


# Imports

In [ ]:
import sys, os
sys.path.insert(0, REPO_PATH)

import warnings, logging, traceback, csv, pickle, time
from datetime import datetime
warnings.filterwarnings('ignore')
logging.getLogger('recbole').setLevel(logging.ERROR)

import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger
from logging import getLogger

from src.utils import path_builder
from src.models.Baselines.GRU4Rec import GRU4Rec
from src.models.Baselines.SASRec import SASRec
from src.models.Baselines.CL4SRec import CL4SRec
from src.models.Baselines.DouRec import DuoRec
from src.models.Baselines.mamba4rec import Mamba4Rec
from src.models.Baselines.gated_mamba import SIGMA
from src.models.SSM_CL.mamba4rec_cl import Mamba4Rec_CL
from src.models.SSM_CL.SIGMA_cl import SIGMA_CL

print('All imports successful')

All imports successful


# Stage Selection
Set STAGE before running the training cell.
- `'1'`  — Base SSMs: Mamba4Rec CE/BPR + SIGMA CE/BPR
- `'2'` — Mamba4Rec_CL CE: InfoNCE + DCL
- `'3'` — Mamba4Rec_CL BPR: InfoNCE + DCL
- `'4'` — SIGMA_CL CE: InfoNCE + DCL
- `'5'` — SIGMA_CL BPR: InfoNCE + DCL
- `'6'`  — Baselines: GRU4Rec, SASRec, CL4SRec, DuoRec

In [ ]:
STAGE = '1'

CONFIG_DIR        = path_builder('src/configs')
DATASET_CONFIG    = path_builder(CONFIG_DIR + '/dataset.yaml')
TRAINING_CONFIG   = path_builder(CONFIG_DIR + '/training.yaml')
MODELS_CONFIG_DIR = path_builder(CONFIG_DIR + '/models')
CSV_PATH          = os.path.join(DRIVE_RESULTS_DIR, f'stage{STAGE}_results.csv')

STAGE_EXPERIMENTS = {
    '1': [
        (Mamba4Rec, 'Mamba4Rec', 'mamba4rec', 'CE',  None),
        (Mamba4Rec, 'Mamba4Rec', 'mamba4rec', 'BPR', None),
        (SIGMA,     'SIGMA',     'sigma',     'CE',  None),
        (SIGMA,     'SIGMA',     'sigma',     'BPR', None),
    ],
    '2': [
        (Mamba4Rec_CL, 'Mamba4Rec_CL', 'mamba4rec_cl', 'CE', 'info_nce'),
        (Mamba4Rec_CL, 'Mamba4Rec_CL', 'mamba4rec_cl', 'CE', 'dcl'),
    ],
    '3': [
        (Mamba4Rec_CL, 'Mamba4Rec_CL', 'mamba4rec_cl', 'BPR', 'info_nce'),
        (Mamba4Rec_CL, 'Mamba4Rec_CL', 'mamba4rec_cl', 'BPR', 'dcl'),
    ],
    '4': [
        (SIGMA_CL, 'SIGMA_CL', 'sigma_cl', 'CE', 'info_nce'),
        (SIGMA_CL, 'SIGMA_CL', 'sigma_cl', 'CE', 'dcl'),
    ],
    '5': [
        (SIGMA_CL, 'SIGMA_CL', 'sigma_cl', 'BPR', 'info_nce'),
        (SIGMA_CL, 'SIGMA_CL', 'sigma_cl', 'BPR', 'dcl'),
    ],
    '6': [
        (GRU4Rec, 'GRU4Rec', 'gru4rec', None, None),
        (SASRec,  'SASRec',  'sasrec',  None, None),
        (DuoRec,  'DuoRec',  'duorec',  None, None),
        (CL4SRec, 'CL4SRec', 'cl4srec', None, None),
    ],
}

STAGE_LABELS = {
    '1':  'Base SSMs (Mamba4Rec + SIGMA)',
    '2': 'Mamba4Rec_CL CE (InfoNCE + DCL)',
    '3': 'Mamba4Rec_CL BPR (InfoNCE + DCL)',
    '4': 'SIGMA_CL CE (InfoNCE + DCL)',
    '5': 'SIGMA_CL BPR (InfoNCE + DCL)',
    '6':  'Baselines',
}

assert STAGE in STAGE_EXPERIMENTS, f'STAGE must be one of {list(STAGE_EXPERIMENTS.keys())}'
EXPERIMENTS = STAGE_EXPERIMENTS[STAGE]

print(f'Stage        : {STAGE} — {STAGE_LABELS[STAGE]}')
print(f'Experiments  : {len(EXPERIMENTS)} per dataset')
print(f'Datasets     : {DATASETS}')
print(f'Total        : {len(EXPERIMENTS) * len(DATASETS)}')
print(f'CSV          : {CSV_PATH}')
print(f'Checkpoints  : {DRIVE_CHECKPOINT_DIR}')

Stage        : 1 — Base SSMs (Mamba4Rec + SIGMA)
Experiments  : 4 per dataset
Datasets     : ['amazon_videogames', 'amazon_toys_and_games', 'movielens_1m', 'lastfm_1k']
Total        : 16
CSV          : /content/drive/MyDrive/results/stage1_results.csv
Checkpoints  : /content/drive/MyDrive/checkpoints


### Geometry Analysis

In [ ]:
def compute_isotropy(embeddings: torch.Tensor):
    embeddings = embeddings - embeddings.mean(dim=0)
    cov = torch.mm(embeddings.T, embeddings) / embeddings.shape[0]
    eigenvalues = torch.linalg.eigvalsh(cov).clamp(min=1e-10)
    eigenvalues = eigenvalues / eigenvalues.sum()
    entropy = -(eigenvalues * torch.log(eigenvalues)).sum()
    return (torch.exp(entropy) / embeddings.shape[1]).item()

def compute_effective_rank(embeddings: torch.Tensor):
    embeddings = embeddings - embeddings.mean(dim=0)
    _, singular_values, _ = torch.linalg.svd(embeddings, full_matrices=False)
    singular_values = singular_values.clamp(min=1e-10)
    singular_values = singular_values / singular_values.sum()
    entropy = -(singular_values * torch.log(singular_values)).sum()
    return torch.exp(entropy).item()

def extract_embeddings(model, n_items: int):
    with torch.no_grad():
        return model.item_embedding.weight[:n_items].cpu().float()

### Custom Trainer with Timing

In [ ]:
class TimedTrainer(Trainer):
    def __init__(self, config, model):
        super().__init__(config, model)
        self.epoch_times = []
        self.train_loss_history = []
        self.valid_metric_history = []
        self.total_train_time = 0

    def fit(self, train_data, valid_data=None, saved=True, show_progress=False):
        self.total_train_time = 0
        self.epoch_times = []
        self.train_loss_history = []
        self.valid_metric_history = []
        best_score, best_result = super().fit(train_data, valid_data, saved, show_progress)
        return best_score, best_result

    def _train_epoch(self, train_data, epoch_idx, loss_func=None, show_progress=False):
        epoch_start = time.perf_counter()
        loss = super()._train_epoch(train_data, epoch_idx, loss_func, show_progress)
        epoch_time = time.perf_counter() - epoch_start

        self.epoch_times.append(epoch_time)
        self.train_loss_history.append(loss)
        self.total_train_time += epoch_time

        if hasattr(self, 'best_valid_score') and self.best_valid_score is not None:
            self.valid_metric_history.append(self.best_valid_score)

        return loss

### Experiment Runner

In [ ]:
def run_unified_experiment(model_class, model_name, config_file, dataset_name,
                           loss_type=None, cl_loss_type=None, skip_if_exists=True):
    exp_id = make_exp_id(model_name, loss_type, cl_loss_type, dataset_name)
    checkpoint_path = os.path.join(DRIVE_CHECKPOINT_DIR, f'{exp_id}.pkl')

    if skip_if_exists and os.path.exists(checkpoint_path):
        print(f"{exp_id} exists - skipping")
        return None

    print(f"  {exp_id}")
    print(f"  Started: {datetime.now().strftime('%H:%M:%S')}")

    result = {
        'exp_id': exp_id, 'model': model_name, 'loss_type': loss_type or 'default',
        'cl_loss_type': cl_loss_type or 'none', 'dataset': dataset_name,
        'status': 'failed', 'error': '',
        'total_train_time_sec': None, 'total_train_time_min': None, 'avg_epoch_time_sec': None,
        'num_epochs': None, 'early_stopped': None, 'best_epoch': None,
        'best_valid_score': None, 'hit@5': None, 'hit@10': None, 'hit@20': None,
        'ndcg@5': None, 'ndcg@10': None, 'ndcg@20': None, 'mrr@5': None, 'mrr@10': None, 'mrr@20': None,
        'isotropy': None, 'effective_rank': None, 'embedding_dim': None, 'n_items': None,
        'epoch_times': [], 'train_losses': [], 'valid_metrics': [],
    }

    try:
        config_dict = {}
        if loss_type:     config_dict['loss_type'] = loss_type
        if cl_loss_type:  config_dict['cl_loss_type'] = cl_loss_type
        if loss_type == 'BPR':
            config_dict['train_neg_sample_args'] = {
                'distribution': 'uniform', 'sample_num': 1,
                'alpha': 1.0, 'dynamic': False, 'candidate_num': 0
            }

        config = Config(
            model=model_class,
            dataset=dataset_name,
            config_file_list=[
                DATASET_CONFIG, TRAINING_CONFIG, path_builder(MODELS_CONFIG_DIR + f'/{config_file}.yaml'),
            ],
            config_dict=config_dict,
        )
        config['data_path'] = path_builder('src/datasets/preprocessed')

        init_seed(config['seed'], config['reproducibility'])
        init_logger(config)

        dataset = create_dataset(config)
        train_data, valid_data, test_data = data_preparation(config, dataset)

        model = model_class(config, dataset).to(config['device'])
        trainer = TimedTrainer(config, model)

        train_start = time.perf_counter()
        best_valid_score, best_valid_result = trainer.fit(train_data, valid_data, saved=True, show_progress=True)
        total_train_time = time.perf_counter() - train_start

        epoch_times = trainer.epoch_times
        train_losses = trainer.train_loss_history
        valid_metrics = trainer.valid_metric_history
        best_epoch = getattr(trainer, 'best_valid_epoch', len(epoch_times) - 1)
        early_stopped = len(epoch_times) < config['epochs']

        epoch_history = [
            {'epoch': i, 'train_loss': l, 'valid_score': valid_metrics[i] if i < len(valid_metrics) else None}
            for i, l in enumerate(train_losses)
        ]

        with open(checkpoint_path, 'wb') as f:
            pickle.dump({
                'model_state_dict': trainer.model.state_dict(),
                'config': {k: v for k, v in config.final_config_dict.items()},
                'best_valid_score': best_valid_score,
                'best_valid_result': best_valid_result,
                'best_epoch': best_epoch, 'total_epochs': len(epoch_times),
                'early_stopped': early_stopped, 'epoch_history': epoch_history,
                'epoch_times': epoch_times, 'total_train_time': total_train_time,
                'exp_id': exp_id,
            }, f)

        print(f"\n  Evaluating on test set...")
        test_result = trainer.evaluate(test_data, load_best_model=False, show_progress=False)

        print(f"  Computing geometry...")
        embeddings = extract_embeddings(model, config['n_items'] if 'n_items' in config else dataset.item_num)
        isotropy = compute_isotropy(embeddings)
        eff_rank = compute_effective_rank(embeddings)

        result.update({
            'status': 'success', 'total_train_time_sec': total_train_time,
            'total_train_time_min': total_train_time / 60,
            'avg_epoch_time_sec': np.mean(epoch_times) if epoch_times else None,
            'num_epochs': len(epoch_times), 'early_stopped': early_stopped, 'best_epoch': best_epoch,
            'best_valid_score': best_valid_score,
            'hit@5': test_result.get('hit@5'), 'hit@10': test_result.get('hit@10'), 'hit@20': test_result.get('hit@20'),
            'ndcg@5': test_result.get('ndcg@5'), 'ndcg@10': test_result.get('ndcg@10'), 'ndcg@20': test_result.get('ndcg@20'),
            'mrr@5': test_result.get('mrr@5'), 'mrr@10': test_result.get('mrr@10'), 'mrr@20': test_result.get('mrr@20'),
            'isotropy': isotropy, 'effective_rank': eff_rank,
            'embedding_dim': embeddings.shape[1], 'n_items': embeddings.shape[0],
            'epoch_times': epoch_times, 'train_losses': train_losses, 'valid_metrics': valid_metrics,
        })

        print(f"\n  Complete")
        print(f"  NDCG@10: {result['ndcg@10']:.4f} | Hit@10: {result['hit@10']:.4f}")
        print(f"  Isotropy: {result['isotropy']:.4f} | Eff. Rank: {result['effective_rank']:.2f}/{result['embedding_dim']}")
        print(f"  Training: {result['total_train_time_min']:.1f} min ({result['avg_epoch_time_sec']:.0f}s/epoch)")
    except Exception as e:
        result['error'] = str(e)
        print(f"  FAILED: {e}")
        traceback.print_exc()

    return result

### CSV Helpers & Plotting Definitions

In [ ]:
RESULTS_HEADERS = [
    'exp_id', 'model', 'loss_type', 'cl_loss_type', 'dataset', 'status', 'error',
    'total_train_time_min', 'avg_epoch_time_sec', 'num_epochs', 'early_stopped', 'best_epoch', 'best_valid_score',
    'hit@5', 'hit@10', 'hit@20', 'ndcg@5', 'ndcg@10', 'ndcg@20', 'mrr@5', 'mrr@10', 'mrr@20',
    'isotropy', 'effective_rank', 'embedding_dim', 'n_items',
]

def save_results(results, path):
    rows = []
    for r in results:
        if not r: continue
        row = {k: v for k, v in r.items() if k not in ['epoch_times', 'train_losses', 'valid_metrics']}
        for k in ['epoch_times', 'train_losses', 'valid_metrics']:
            if k in r and r[k]: row[k] = str(r[k])[:1000]
        rows.append(row)
    with open(path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=RESULTS_HEADERS)
        writer.writeheader()
        writer.writerows(rows)
    print(f"\nResults saved → {path}")

DATASET_LABELS = {'amazon_videogames': 'Video Games', 'amazon_toys_and_games': 'Toys & Games', 'movielens_1m': 'ML-1M', 'lastfm_1k': 'LastFM-1K'}
MODEL_COLORS = {
    'Mamba4Rec_CE': '#F59E0B', 'Mamba4Rec_BPR': '#D97706', 'Mamba4Rec_CL_CE_info_nce': '#EF4444', 'Mamba4Rec_CL_CE_dcl': '#DC2626',
    'SIGMA_CE': '#10B981', 'SIGMA_BPR': '#059669', 'SIGMA_CL_CE_info_nce': '#34D399', 'SIGMA_CL_CE_dcl': '#6EE7B7',
    'GRU4Rec': '#6B7280', 'SASRec': '#3B82F6', 'CL4SRec': '#06B6D4', 'DuoRec': '#8B5CF6',
}

def get_color(label): return MODEL_COLORS.get(label, '#9CA3AF')
def model_label(row):
    parts = [row['model']]
    if row['loss_type'] != 'default': parts.append(row['loss_type'])
    if row['cl_loss_type'] != 'none': parts.append(row['cl_loss_type'])
    return '_'.join(parts)

def plot_convergence(results, output_dir):
    successful = [r for r in results if r and r['status'] == 'success' and r.get('train_losses')]
    for r in successful:
        exp_id = r['exp_id']
        epochs = list(range(len(r['train_losses'])))
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.plot(epochs, r['train_losses'], 'b-', linewidth=1.5)
        ax1.set_title(f'{exp_id} — Training Loss')
        if r.get('valid_metrics'):
            ax2.plot(list(range(len(r['valid_metrics']))), r['valid_metrics'], 'g-', linewidth=1.5)
            ax2.axhline(y=r.get('best_valid_score', 0), color='r', linestyle='--')
        ax2.set_title(f'{exp_id} — Validation')
        plt.savefig(os.path.join(output_dir, f'convergence_{exp_id}.png'), dpi=100, bbox_inches='tight')
        plt.close()

def plot_metric_comparison(results, metric, title, output_path):
    successful = [r for r in results if r and r['status'] == 'success' and r.get(metric) is not None]
    if not successful: return
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(20, 6), sharey=True)
    if len(DATASETS) == 1: axes = [axes]
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    for ax, dataset in zip(axes, DATASETS):
        ds_results = [r for r in successful if r['dataset'] == dataset]
        if not ds_results: ax.set_visible(False); continue
        labels = [model_label(r) for r in ds_results]
        values = [float(r[metric]) for r in ds_results]
        bars = ax.bar(range(len(labels)), values, color=[get_color(l) for l in labels])
        ax.set_title(DATASET_LABELS.get(dataset, dataset))
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()

def plot_timing_comparison(results, output_path):
    plot_metric_comparison(results, 'total_train_time_min', 'Training Time by Model and Dataset', output_path)

# Run Training and Evaluation

In [ ]:

plots_dir = os.path.join(DRIVE_RESULTS_DIR, 'plots')
os.makedirs(plots_dir, exist_ok=True)


print(f"  Training & Evaluation")
print(f"  Stage {STAGE}: {STAGE_LABELS[STAGE]}")
print(f"  Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


all_results = []
global_start = time.perf_counter()

for dataset in DATASETS:
    print(f"\n  DATASET: {dataset}\n")
    for model_class, model_name, config_file, loss_type, cl_loss_type in EXPERIMENTS:
        result = run_unified_experiment(
            model_class=model_class,
            model_name=model_name,
            config_file=config_file,
            dataset_name=dataset,
            loss_type=loss_type,
            cl_loss_type=cl_loss_type,
            skip_if_exists=True
        )
        if result:
            all_results.append(result)

total_time = (time.perf_counter() - global_start) / 60
csv_path = os.path.join(DRIVE_RESULTS_DIR, f'stage{STAGE}_results_full.csv')
save_results(all_results, csv_path)

print(f"\nGenerating plots...")
plot_convergence(all_results, plots_dir)
plot_metric_comparison(all_results, 'ndcg@10', 'NDCG@10 by Model and Dataset', os.path.join(plots_dir, f'stage{STAGE}_ndcg10.png'))
plot_metric_comparison(all_results, 'hit@10', 'Hit@10 by Model and Dataset', os.path.join(plots_dir, f'stage{STAGE}_hit10.png'))
plot_metric_comparison(all_results, 'mrr@10', 'MRR@10 by Model and Dataset', os.path.join(plots_dir, f'stage{STAGE}_mrr10.png'))
plot_timing_comparison(all_results, os.path.join(plots_dir, f'stage{STAGE}_training_time.png'))

print(f"\n  Results: {csv_path}")
print(f"  Plots: {plots_dir}")
